# Traffic Flow Optimization Formulation for a Simplified 2-Phase Signal

This notebook documents the optimization problem that will guide later traffic signal timing analysis in the capstone project. The focus here is formulation only: defining the traffic signal setting, optimization objective, decision variables, inputs, and operational constraints in a clean and interpretable form.


## 1. Title and Objective

The objective of this notebook is to formally define a simplified traffic flow optimization problem for improving intersection efficiency using data-driven signal timing. The formulation establishes the mathematical structure that will later support computational signal timing optimization, while remaining concise, interpretable, and appropriate for a capstone-level study.


## 2. Define the Traffic Signal Setting

The optimization setting assumes a simplified 2-phase signalized intersection:

- **Phase 1** serves North-South traffic.
- **Phase 2** serves East-West traffic.

This abstraction is intentional. A 2-phase model reduces operational complexity while preserving the core tradeoff of signal timing control: limited green time must be allocated across competing traffic movements. For the capstone project, this simplified structure makes the optimization problem both interpretable and tractable.


## 3. Define the Optimization Goal

The primary optimization goal is to **minimize average vehicle delay** across the intersection. Delay minimization is appropriate because it directly reflects operational inefficiency experienced by road users and provides a clear metric for comparing alternative signal timing plans.

Other reasonable objectives include reducing queue lengths, improving throughput, or balancing service equity across approaches. However, this formulation adopts delay minimization as the principal objective because it is both academically standard and well aligned with the capstone's traffic flow efficiency focus.


## 4. Define Optimization Inputs

The optimization model requires a compact set of operational and demand-related inputs. These values summarize traffic conditions and define the feasible timing environment for the simplified signal control problem.

| Input | Symbol | Units | Description |
| --- | --- | --- | --- |
| Traffic demand for Phase 1 | $v_1$ | veh/hr | Expected North-South demand during the analysis period |
| Traffic demand for Phase 2 | $v_2$ | veh/hr | Expected East-West demand during the analysis period |
| Arrival rate for Phase 1 | $\lambda_1$ | veh/hr | Estimated arrival intensity for Phase 1 |
| Arrival rate for Phase 2 | $\lambda_2$ | veh/hr | Estimated arrival intensity for Phase 2 |
| Saturation flow rate | $s$ | veh/hr of green | Maximum service flow during effective green |
| Cycle length | $C$ | sec | Total duration of one complete signal cycle |
| Peak-period traffic estimate | $q_{\text{peak}}$ | veh/hr | Representative demand level during congested conditions |

The table below presents the same inputs as a small pandas dataframe for later computational use.


In [1]:
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 120)

optimization_inputs = pd.DataFrame(
    [
        {'Input': 'Traffic demand for Phase 1', 'Symbol': 'v1', 'Units': 'veh/hr', 'Description': 'Expected North-South traffic demand'},
        {'Input': 'Traffic demand for Phase 2', 'Symbol': 'v2', 'Units': 'veh/hr', 'Description': 'Expected East-West traffic demand'},
        {'Input': 'Arrival rate for Phase 1', 'Symbol': 'lambda1', 'Units': 'veh/hr', 'Description': 'Arrival intensity for Phase 1'},
        {'Input': 'Arrival rate for Phase 2', 'Symbol': 'lambda2', 'Units': 'veh/hr', 'Description': 'Arrival intensity for Phase 2'},
        {'Input': 'Saturation flow rate', 'Symbol': 's', 'Units': 'veh/hr of green', 'Description': 'Maximum service rate during effective green'},
        {'Input': 'Cycle length', 'Symbol': 'C', 'Units': 'sec', 'Description': 'Total signal cycle duration'},
        {'Input': 'Peak-period traffic estimate', 'Symbol': 'q_peak', 'Units': 'veh/hr', 'Description': 'Representative peak-period demand level'},
    ]
)

display(optimization_inputs)


,Input,Symbol,Units,Description
0,Traffic demand for Phase 1,v1,veh/hr,Expected North-South traffic demand
1,Traffic demand for Phase 2,v2,veh/hr,Expected East-West traffic demand
2,Arrival rate for Phase 1,lambda1,veh/hr,Arrival intensity for Phase 1
3,Arrival rate for Phase 2,lambda2,veh/hr,Arrival intensity for Phase 2
4,Saturation flow rate,s,veh/hr of green,Maximum service rate during effective green
5,Cycle length,C,sec,Total signal cycle duration
6,Peak-period traffic estimate,q_peak,veh/hr,Representative peak-period demand level


## 5. Define Decision Variables

The optimization problem controls three signal timing variables:

- $g_1$: green time allocated to Phase 1
- $g_2$: green time allocated to Phase 2
- $C$: total signal cycle length

These quantities are the controllable parameters of the optimization problem. Adjusting them changes how available green time is distributed between North-South and East-West traffic and therefore affects intersection delay.


In [2]:
decision_variables = pd.DataFrame(
    [
        {'Variable': 'g1', 'Units': 'sec', 'Interpretation': 'Green time allocated to Phase 1 (North-South)'},
        {'Variable': 'g2', 'Units': 'sec', 'Interpretation': 'Green time allocated to Phase 2 (East-West)'},
        {'Variable': 'C', 'Units': 'sec', 'Interpretation': 'Total cycle length'},
    ]
)

display(decision_variables)


,Variable,Units,Interpretation
0,g1,sec,Green time allocated to Phase 1 (North-South)
1,g2,sec,Green time allocated to Phase 2 (East-West)
2,C,sec,Total cycle length


## 6. Specify Operational Constraints

The timing plan must satisfy practical operating constraints so that the optimization remains realistic.

- The allocated green times cannot exceed the total cycle length: $g_1 + g_2 \leq C$.
- Each phase must receive at least a minimum green time: $g_1 \geq g_{\min}$ and $g_2 \geq g_{\min}$.
- The cycle length must remain within an operational upper bound: $C \leq C_{\max}$.

For illustration, this notebook adopts $g_{\min} = 15$ seconds and $C_{\max} = 120$ seconds. These limits help preserve safe and functional signal timing. The slack allowed by $g_1 + g_2 \leq C$ can be interpreted as time reserved for clearance intervals or other lost time not modeled explicitly in this simplified formulation.


In [3]:
constraints = pd.DataFrame(
    [
        {'Constraint': 'g1 + g2 <= C', 'Illustrative value': 'Feasibility condition', 'Purpose': 'Total green allocation must fit within the cycle'},
        {'Constraint': 'g1 >= g_min', 'Illustrative value': 'g_min = 15 sec', 'Purpose': 'Ensures minimum service for Phase 1'},
        {'Constraint': 'g2 >= g_min', 'Illustrative value': 'g_min = 15 sec', 'Purpose': 'Ensures minimum service for Phase 2'},
        {'Constraint': 'C <= C_max', 'Illustrative value': 'C_max = 120 sec', 'Purpose': 'Prevents excessively long cycles'},
    ]
)

display(constraints)


,Constraint,Illustrative value,Purpose
0,g1 + g2 <= C,Feasibility condition,Total green allocation must fit within the cycle
1,g1 >= g_min,g_min = 15 sec,Ensures minimum service for Phase 1
2,g2 >= g_min,g_min = 15 sec,Ensures minimum service for Phase 2
3,C <= C_max,C_max = 120 sec,Prevents excessively long cycles


## 7. Mathematical Formulation

The simplified optimization problem can be expressed as follows.

**Objective:**

$$
\min_{g_1, g_2, C} \; D = d_1 + d_2
$$

where:

- $D$ is the total average delay across the intersection,
- $d_1$ is the average delay associated with Phase 1,
- $d_2$ is the average delay associated with Phase 2.

For each phase $i \in \{1, 2\}$, delay is represented conceptually as a function of demand, capacity, and allocated green time:

$$
d_i = f(v_i, s, g_i, C)
$$

where $v_i$ denotes traffic demand for phase $i$, $s$ denotes the saturation flow rate, $g_i$ denotes the green time allocated to phase $i$, and $C$ denotes the total cycle length. In later notebooks, this abstract delay function can be translated into a simplified Webster-style delay expression or another comparable operational approximation.

**Constraints:**

$$
g_1 + g_2 \leq C
$$

$$
g_1 \geq g_{\min}
$$

$$
g_2 \geq g_{\min}
$$

$$
C \leq C_{\max}
$$

This formulation is intentionally compact. It captures the central optimization logic needed for later signal timing experiments without introducing unnecessary model complexity at the formulation stage.


## 8. Example Parameter Table

The following example parameter set provides a simple reference scenario for the 2-phase formulation. These values are illustrative and are intended to support interpretation of the optimization setup rather than serve as calibrated field measurements.


In [4]:
example_parameters = pd.DataFrame(
    [
        {'Parameter': 'v1', 'Value': 800, 'Units': 'veh/hr', 'Meaning': 'Phase 1 traffic demand'},
        {'Parameter': 'v2', 'Value': 500, 'Units': 'veh/hr', 'Meaning': 'Phase 2 traffic demand'},
        {'Parameter': 'saturation_flow', 'Value': 1800, 'Units': 'veh/hr', 'Meaning': 'Saturation flow rate'},
        {'Parameter': 'cycle_length', 'Value': 90, 'Units': 'sec', 'Meaning': 'Illustrative cycle length'},
        {'Parameter': 'minimum_green', 'Value': 15, 'Units': 'sec', 'Meaning': 'Minimum green per phase'},
    ]
)

display(example_parameters)


,Parameter,Value,Units,Meaning
0,v1,800,veh/hr,Phase 1 traffic demand
1,v2,500,veh/hr,Phase 2 traffic demand
2,saturation_flow,1800,veh/hr,Saturation flow rate
3,cycle_length,90,sec,Illustrative cycle length
4,minimum_green,15,sec,Minimum green per phase


## 9. Implementation Plan and Next Steps

Subsequent notebooks will build directly on this formulation by translating the objective and constraints into a computational optimization model. That later work will focus on solving for green time allocation, comparing optimized timing against a baseline timing plan, and evaluating whether optimization improves traffic flow efficiency under forecasted demand conditions.

Accordingly, this notebook serves as the formal problem-definition stage of the capstone's signal timing optimization workflow.
